# Hexagonal boundary-layer problem

Araya et al. (2024), section 5.2.2: epsilon=0.01, advection=(1,0), f=1, local P3 and face P1. The exact profile, published figures and separate macro/skeleton/local refinements use explicit original meshes.

The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/11c9fa130acedd94170a0c07bc3f4e24aadfac6f74e0edeebade1e943dd7b506/46_hexagonal_boundary_layer-companion.zip"
COMPANION_SHA256 = "11c9fa130acedd94170a0c07bc3f4e24aadfac6f74e0edeebade1e943dd7b506"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("transport/46_hexagonal_boundary_layer.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"


In [ ]:
from pathlib import Path
import json
from IPython.display import Image, display
record = json.loads((root / "examples/results/rad-layer.json").read_text())
rows = record["records"]
assert len(rows) == 15
assert all(r["residual"] < 1e-10 for r in rows)
[(r["study"], r["n"], r["segments"], r["free_global_dofs"], r["error_v"]) for r in rows]

The V norm is the broken gradient norm plus one half the squared L2 norm. It is distinct from the diffusion-weighted seminorm. DOF comparisons exclude prescribed exterior coefficients; total dimensions are archived separately. Local refinement at a fixed trace space reaches a nonzero approximation floor.

In [ ]:
for name in ("fields", "profile", "elevation", "convergence", "published-norm"):
    display(Image(filename=str(root / f"docs/figures/rad-layer/{name}.png")))
if HISTORICAL_REPLAY:
    for name in ("published-figure5", "published-figure6"):
        display(Image(filename=str(root / f"docs/figures/rad-layer/{name}.png")))
else:
    print("Attributed article rasters are optional historical assets.")


Independent DOLFINx/UFL complete-system comparisons use the same material, boundary conditions and local/skeletal spaces. Saved coordinates, connectivity and polynomial ordering permit field replay. The native code does not reuse PyMHM local matrices or condensation. Historical connectivity remains unspecified.

In [ ]:
if HISTORICAL_REPLAY:
    import hashlib
    native = json.loads((root / "examples/results/rad-native/verification.json").read_text())
    assert native["accepted"]
    native_rows = [row for row in native["rows"] if row["kind"] == "layer"]
    assert len(native_rows) == 3
    for row in native_rows:
        archive = root / "examples/results/rad-native" / row["archive"]
        assert hashlib.sha256(archive.read_bytes()).hexdigest() == row["archive_sha256"]
        assert row["accepted"]
    [(row["n"], row["local_refinement"], row["case"], row["epsilon"],
      row["physical_norms"]["20"]["physical_flux"]["relative_difference"])
     for row in native_rows]
else:
    print("Original external archive replay is optional; use --historical with its attributed payloads.")


In [ ]:
if HISTORICAL_REPLAY:
    display(Image(filename=str(root / "docs/figures/rad-layer/native-comparison.png")))
else:
    print("Original external archive replay is optional; use --historical with its attributed payloads.")
